In [149]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [150]:
# !pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [151]:
# %pip install umap-learn

In [152]:
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import os
from google import genai
import numpy as np
from pydantic import BaseModel, Field
from google.genai.types import GenerateContentConfig
import requests

class TrendInsight(BaseModel):
    trend_name: str = Field(description="A catchy 2-to-4 word label for the trend.")
    key_ingredients_or_products: list[str] = Field(description="Specific products, ingredients, or tools explicitly mentioned in the posts.")
    consumer_pain_point: str = Field(description="The underlying problem or insecurity the consumers are trying to solve.")
    capitalization_strategy: str = Field(description="A 1-sentence idea on how a brand could capitalize on this specific trend. Make it directand actionable.")
    actionability_score: int = Field(description="A score from 1-10 on how easily a business could monetize this trend.")

class KeywordResponse(BaseModel):
    keywords: list[str] = Field(description="A list of semantically related words.")

# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [153]:
def get_related_keywords(primary_word: str, num_related: int = 4) -> list[str]:
    """Queries Gemini for consumer-behavior search terms related to `primary_word`."""
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    prompt = f"""
    You are an expert consumer behavior analyst focused on predicting emerging trends and discovering market whitespace. 
    I am building a search query to find organic, everyday conversations about '{primary_word}' on social media.
    
    Generate exactly {num_related} highly relevant search terms that capture the context where new trends emerge.
    
    To find whitespace BEFORE a trend happens, your terms must focus on:
    - Consumer pain points, struggles, or complaints (e.g., "damaged", "soreness", "too expensive")
    - Daily routines, habits, or generic goals (e.g., "morning routine", "hydration", "recovery")
    - Unmet needs or DIY workarounds
    
    STRICT RULES:
    - ABSOLUTELY NO brand names or specific product names.
    - ABSOLUTELY NO existing viral trend names or catchy social media slang (we want the raw behaviors that precede trends).
    - STAY DOMAIN-ANCHORED: Keep terms strictly within the direct ecosystem of '{primary_word}'. Do not drift into broad, unrelated macro-topics.
       - Example for 'stocks': Good terms are "market dip", "first investment", "portfolio loss", "trading app". Bad terms are "paying rent" or "grocery bill" (too far removed).
    - Keep terms short (1 to 2 words maximum).
    - Do NOT include the '#' symbol.
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=KeywordResponse,
            )
        )
        
        related_words = response.parsed.keywords
        return related_words[:num_related]
        
    except Exception as e:
        print(f"Gemini API error: {e}")
        return []

In [154]:
def fetch_bluesky_posts(query, target_count):

    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"],
                "replyCount": post.get("replyCount", 0),
                "repostCount": post.get("repostCount", 0),
                "likeCount": post.get("likeCount", 0),
                "quoteCount": post.get("quoteCount", 0)
                # "has_embed_link": has_embed,
                # "labels": labels
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_posts.head()

In [155]:
def filter_spam_posts(df, threshold):
    """
    Calculates a spam score (0.0 to 1.0) based on engagement, duplication, 
    and text formatting, then filters out posts above the threshold.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_scored = df.copy()
    
    # Initialize score
    df_scored['spam_score'] = 0.0
    
    # 1. Duplication Penalty (Strongest signal)
    # Flag posts that share the exact same text (e.g., cross-posting bots)
    is_duplicate = df_scored.duplicated(subset=['text'], keep='first')
    df_scored.loc[is_duplicate, 'spam_score'] += 0.5
    
    # 2. Low Engagement Penalty
    # Summing up the engagement metrics you are already fetching
    df_scored['total_engagement'] = (
        df_scored['replyCount'] + 
        df_scored['repostCount'] + 
        df_scored['likeCount'] + 
        df_scored['quoteCount']
    )
    # Add a penalty if the post has completely zero engagement
    df_scored.loc[df_scored['total_engagement'] == 0, 'spam_score'] += 0.2
    
    # 3. Content Heuristics (Links & Hashtags)
    # Count occurrences using basic regex
    df_scored['hashtag_count'] = df_scored['text'].str.count(r'#\w+')
    df_scored['link_count'] = df_scored['text'].str.count(r'http[s]?://')
    
    # Penalize spammy text formatting
    df_scored.loc[df_scored['hashtag_count'] > 4, 'spam_score'] += 0.15
    df_scored.loc[df_scored['link_count'] >= 2, 'spam_score'] += 0.15
    
    # 4. Cap the maximum score at 1.0
    df_scored['spam_score'] = df_scored['spam_score'].clip(upper=1.0)
    
    # Filter the DataFrame based on the acceptable threshold
    initial_count = len(df_scored)
    df_filtered = df_scored[df_scored['spam_score'] < threshold].copy()
    filtered_count = len(df_filtered)
    
    print(f"Filtered out {initial_count - filtered_count} spam-likely posts.")
    
    # Clean up calculation columns before passing to the clustering phase
    df_filtered = df_filtered.drop(columns=['total_engagement', 'hashtag_count', 'link_count'])
    
    return df_filtered


# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_clean = filter_spam_posts(df_posts, threshold=0.6)
# df_clustered = cluster_social_posts(df_clean)

In [156]:
# df_clean

In [157]:
def cluster_social_posts(df, cluster_fraction, sample_fraction):
    print("Loading Sentence Transformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    print("Generating embeddings...")
    embeddings = model.encode(df['text'].tolist())
    
    print("Reducing dimensions with UMAP...")
    # Compress the 384 dimensions down to 5 to help HDBSCAN find density
    umap_model = umap.UMAP(
        n_neighbors=30, # Focuses on local neighborhood size 
        n_components=5, # Reduce to 5 dimensions
        min_dist=0.0,   # How tightly to pack points together 
        metric='cosine',# Cosine works best for text embeddings
        random_state=42 # Ensure reproducible results
    )
    reduced_embeddings = umap_model.fit_transform(embeddings)
    
    print("Running HDBSCAN clustering...")
    
    # Calculate dynamic parameters based on DataFrame size
    total_posts = len(df)
    
    # Force the values to be integers, and set an absolute minimum floor (e.g., 5)
    # so small datasets don't end up with a min_cluster_size of 1.
    dynamic_min_cluster_size = max(5, int(total_posts * cluster_fraction))
    dynamic_min_samples = max(5, int(dynamic_min_cluster_size * 0.5))
    
    print(f"Dynamic Settings: min_cluster_size={dynamic_min_cluster_size}, min_samples={dynamic_min_samples}")
    
    print("Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=dynamic_min_cluster_size, 
        min_samples=dynamic_min_samples,      
        metric='euclidean'
        # , cluster_selection_epsilon=0.05  
    )


    df['cluster_id'] = clusterer.fit_predict(reduced_embeddings)
    
    # -1 means "noise" (unclustered). Let's filter those out.
    clustered_df = df[df['cluster_id'] != -1]
    
    print(f"Found {len(clustered_df['cluster_id'].unique())} unique clusters.")
    return clustered_df
# Test
# df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
# print(df_clustered['cluster_id'].value_counts())

In [158]:
# # Test keywords

# # lst_keywords = ["skincare", "toothpaste", "haircare", "electric vehicle", "mens fashion", "stocks", "investment", "fitness"]

# lst_keywords = ["music"]

# for keyword in lst_keywords:
#     print(keyword)
#     df_posts = fetch_bluesky_posts(keyword, target_count=5000)
#     df_clean = filter_spam_posts(df_posts, threshold=0.1)
#     df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
#     print(df_clustered['cluster_id'].value_counts())
#     # df_clustered = pd.DataFrame()



In [159]:
def extract_actionable_insights(df_clustered):
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    # Create a dictionary to hold our rich insights
    cluster_insights = {}
    # Count the posts in each cluster and sort them from largest to smallest
    cluster_sizes = df_clustered['cluster_id'].value_counts()
    
    # Grab the IDs of the top 6 largest clusters
    top_6_clusters = cluster_sizes.head(6).index.tolist()
    
# Iterate ONLY over those top 6
    for cluster_id in top_6_clusters:
        
        # Isolate the dataframe to just the posts in the current cluster
        cluster_df = df_clustered[df_clustered['cluster_id'] == cluster_id].copy()
        
        # Calculate total interactions for these specific posts
        cluster_df['total_interactions'] = (
            cluster_df['likeCount'] + 
            cluster_df['repostCount'] + 
            cluster_df['replyCount'] + 
            cluster_df['quoteCount']
        )
        
        # Sort by the most interacted posts first, then grab the top 15
        sample_posts = cluster_df.sort_values(by='total_interactions', ascending=False)['text'].head(15).tolist()

        posts_text = "\n- ".join(sample_posts)
        
        prompt = f"""
        You are an expert consumer trend analyst and product developer. 
        Analyze the following social media posts that have been clustered together:
        - {posts_text}
        
        Extract the underlying trend and identify exactly how a business can capitalize on it.
        """
        
        # Enforce structured output via GenerateContentConfig
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TrendInsight,
            )
        )
        
        # Access the structured data safely using .parsed
        insight = response.parsed
        cluster_insights[cluster_id] = insight
        
        print(f"Analyzed Cluster {cluster_id}: {insight.trend_name}. \nProduct: {insight.capitalization_strategy} (Score: {insight.actionability_score}/10)")
        
    # Map the new structured data back to the dataframe
    df_clustered['trend_name'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].trend_name if x in cluster_insights else None)
    df_clustered['key_products'] = df_clustered['cluster_id'].map(lambda x: ", ".join(cluster_insights[x].key_ingredients_or_products) if x in cluster_insights else None)
    df_clustered['pain_point'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].consumer_pain_point if x in cluster_insights else None)
    df_clustered['strategy'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].capitalization_strategy if x in cluster_insights else None)
    df_clustered['actionability_score'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].actionability_score if x in cluster_insights else None)
    
    return df_clustered

In [160]:
# 1. Define seed keyword and settings
seed_keyword = "world cup"
posts_per_keyword = 1500 

# 2. Fetch related keywords
print(f"Fetching related keywords for '{seed_keyword}'...")
related_words = get_related_keywords(seed_keyword, num_related = 3)

# 3. Combine the seed keyword with the related words
all_keywords = [seed_keyword] + related_words
all_keywords = [item.replace(" ", "") for item in all_keywords]
print(f"Expanded search keywords: {all_keywords}")


Fetching related keywords for 'world cup'...
Expanded search keywords: ['worldcup', 'matchschedule', 'ticketcost', 'stadiumtravel']


In [161]:

# 4. Fetch posts for all keywords
all_posts = []
for keyword in all_keywords:
    print(f"\n--- Fetching data for: {keyword} ---")
    df_temp = fetch_bluesky_posts(keyword, target_count=posts_per_keyword)
    all_posts.append(df_temp)

# 5. Combine everything into one master DataFrame
df_combined_posts = pd.concat(all_posts, ignore_index=True)
print(f"\nTotal posts collected across all keywords: {len(df_combined_posts)}")

# 6. Run the rest of your pipeline on the combined dataset
print("\n--- Filtering Spam ---")
df_clean = filter_spam_posts(df_combined_posts, threshold = 0.1)

print("\n--- Clustering Posts ---")
df_clustered = cluster_social_posts(df_clean, cluster_fraction = 0.01, sample_fraction = 0.002)




--- Fetching data for: worldcup ---
Fetching 1500 posts for 'worldcup'...
Successfully fetched 1500 posts.

--- Fetching data for: matchschedule ---
Fetching 1500 posts for 'matchschedule'...
Successfully fetched 6 posts.

--- Fetching data for: ticketcost ---
Fetching 1500 posts for 'ticketcost'...
Successfully fetched 0 posts.

--- Fetching data for: stadiumtravel ---
Fetching 1500 posts for 'stadiumtravel'...
Successfully fetched 0 posts.

Total posts collected across all keywords: 1506

--- Filtering Spam ---
Filtered out 1092 spam-likely posts.

--- Clustering Posts ---
Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...
Reducing dimensions with UMAP...


c:\Users\JesusSantillanMinila\Documents\GitHub\social-media-trend-detector\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=5, min_samples=5
Running HDBSCAN clustering...
Found 2 unique clusters.


In [ ]:
print("\n--- Extracting Actionable Insights ---")
df_labeled = extract_actionable_insights(df_clustered)

# View the final structured output
df_labeled


--- Extracting Actionable Insights ---
Analyzed Cluster 1: Fan-First Football Reform. 
Product: Launch an independent, ethically sourced fan apparel brand that transparently channels a percentage of profits back into grassroots community football clubs. (Score: 5/10)
Analyzed Cluster 0: Viral YouTube Shorts. 
Product: Brands should produce bite-sized, humorous, or awe-inspiring short-form video content linked to their products to capture algorithmic reach and drive impulse engagement. (Score: 9/10)


,text,created_at,author,replyCount,repostCount,likeCount,quoteCount,spam_score,cluster_id,trend_name,key_products,pain_point,strategy,actionability_score
15,A new episode is out! This week:\n- #SportingK...,2026-08-03T15:50:07.766Z,fortheglorykc.bsky.social,0,0,1,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
16,A new @fortheglorykc.bsky.social is out! This ...,2026-08-03T15:48:34.813Z,kcsoccerjournal.bsky.social,0,0,1,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
22,Worldcup Rugbyサイトが11年前の南アフリカ戦の興奮をまた新たにUPしてたのでつ...,2026-08-03T13:54:47.767Z,adooda.bsky.social,0,0,2,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
23,www.instagram.com/reel/Da_mnno...,2026-08-03T13:31:41.819Z,hopefulrosey.bsky.social,0,0,1,0,0.0,0,Viral YouTube Shorts,"YouTube Shorts, Instagram Reels","Short attention spans requiring quick, enterta...","Brands should produce bite-sized, humorous, or...",9
24,Mauricio Pochettino will return as the #USMNT ...,2026-08-03T13:31:15.700Z,blazindw.bsky.social,0,1,4,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1485,Trump posted an edit of the video of the World...,2026-07-22T19:27:10.190Z,paidprotester.bsky.social,1,0,0,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
1488,"Sure #Spain isn't racist, with a tradition cal...",2026-07-22T19:14:20.850Z,faithslayer202.bsky.social,0,0,0,1,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
1490,Thanks @barackobama.bsky.social for bringing t...,2026-07-22T19:06:26.036Z,bselected.bsky.social,0,0,1,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
1491,Pico for Martinez and I’d say that’s about rig...,2026-07-22T19:02:34.324Z,davidflane.bsky.social,0,0,2,0,0.0,1,Fan-First Football Reform,"WorldCup jerseys, fan merchandise, political c...","Frustration with corporate greed, political co...","Launch an independent, ethically sourced fan a...",5
